# 🌲 Random Forest Analysis — Education Dataset
### Step-by-step analysis of what factors affect `pisa_score` across countries
---
**Target variable:** `pisa_score` — international education quality benchmark  
**Dataset:** 10 countries × 10 features  
**Goal:** Train a Random Forest, understand feature importance, tune hyperparameters


## 📦 Cell 1 — Import Libraries

In [ ]:
# ─────────────────────────────────────────────────────────────
# IMPORT ALL REQUIRED LIBRARIES
# ─────────────────────────────────────────────────────────────

import pandas as pd                          # DataFrame operations (like Excel in Python)
import numpy as np                           # Numerical computing
import matplotlib.pyplot as plt             # Base plotting library
import seaborn as sns                        # High-level statistical visualizations
import warnings
warnings.filterwarnings('ignore')            # Suppress minor warnings

# ── Scikit-learn: our ML toolkit ──
from sklearn.ensemble import RandomForestRegressor     # The Random Forest model
from sklearn.model_selection import train_test_split   # Split data into train/test
from sklearn.metrics import (
    mean_squared_error,                      # Average squared error
    mean_absolute_error,                     # Average absolute error
    r2_score                                 # Goodness-of-fit (1.0 = perfect, 0 = baseline)
)
from sklearn.inspection import permutation_importance  # More reliable feature importance

# ── Plot style ──
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')

print("✅ All libraries imported successfully!")


## 📂 Cell 2 — Load the Dataset

In [ ]:
# ─────────────────────────────────────────────────────────────
# LOAD CSV INTO A PANDAS DATAFRAME
# ─────────────────────────────────────────────────────────────

# If running locally, update the path to where your CSV is saved.
# If running in Google Colab, upload the file first then use:
#   from google.colab import files
#   uploaded = files.upload()
#   df = pd.read_csv('final_dataset.csv')

df = pd.read_csv('final_dataset.csv')          # ← Change path if needed

# Strip accidental whitespace from column names (common CSV issue)
df.columns = df.columns.str.strip()

print(f"✅ Dataset loaded!")
print(f"📐 Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")
df                                              # Display the full table in Jupyter


## 🔍 Cell 3 — Explore the Data

In [ ]:
# ─────────────────────────────────────────────────────────────
# BASIC DATA EXPLORATION — Always do this before any ML work
# ─────────────────────────────────────────────────────────────

print("=" * 55)
print("📌 DATA TYPES")
print("=" * 55)
print(df.dtypes)

print("\n" + "=" * 55)
print("📌 MISSING VALUES (should all be 0)")
print("=" * 55)
print(df.isnull().sum())

print("\n" + "=" * 55)
print("📌 STATISTICAL SUMMARY")
print("=" * 55)
df.describe().round(2)


## ⚙️ Cell 4 — Define Features (X) and Target (y)

In [ ]:
# ─────────────────────────────────────────────────────────────
# SEPARATE FEATURES FROM TARGET VARIABLE
# ─────────────────────────────────────────────────────────────

# TARGET: The value we want to PREDICT
# employment_outcomes = international education quality score per country
y = df['employment_outcomes']

# FEATURES: The columns we USE to make predictions
# Drop 'country' (text, not usable in ML) and 'employment_outcomes' (that's our answer)
X = df.drop(columns=['country', 'employment_outcomes'])

print(f"Feature matrix X: {X.shape[0]} rows × {X.shape[1]} features")
print(f"Target vector  y: {y.shape[0]} values")
print(f"\nFeatures:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")
print(f"\nTarget range: {y.min()} → {y.max()}, Mean: {y.mean():.1f}")


## 🔥 Cell 5 — Correlation Heatmap

In [ ]:
# ─────────────────────────────────────────────────────────────
# CORRELATION HEATMAP
# Correlation ranges from -1 to +1:
#   +1 = both features go up together (perfect positive)
#   -1 = one goes up, other goes down (perfect negative)
#    0 = no relationship
# ─────────────────────────────────────────────────────────────

# Include employment_outcomes so we can see what predicts it
corr_matrix = df.drop(columns=['country']).corr()

fig, ax = plt.subplots(figsize=(12, 9))

sns.heatmap(
    corr_matrix,
    annot=True,                  # Show correlation numbers in each cell
    fmt='.2f',                   # Round to 2 decimal places
    cmap='RdBu_r',               # Blue = positive, Red = negative
    center=0,                    # Colormap centered at 0 (no correlation)
    linewidths=0.5,              # Grid lines between cells
    square=True,                 # Square cells for readability
    ax=ax,
    annot_kws={"size": 9}
)

ax.set_title('Correlation Heatmap — All Features + employment_outcomes',
             fontsize=14, fontweight='bold', pad=15)
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.tick_params(axis='y', rotation=0,  labelsize=9)

plt.tight_layout()
plt.show()

# ── Print sorted correlations with target ──
print("\n📌 Correlations with employment_outcomes (sorted):")
pisa_corr = corr_matrix['employment_outcomes'].drop('employment_outcomes').sort_values(ascending=False)
for feat, val in pisa_corr.items():
    bar = '█' * int(abs(val) * 20)
    sign = '(+)' if val > 0 else '(-)'
    print(f"  {feat:<30s} {sign}  {val:+.3f}  {bar}")


## 📈 Cell 6 — Feature vs Target Scatter Plots

In [ ]:
# ─────────────────────────────────────────────────────────────
# SCATTER PLOTS: Each feature vs employment_outcomes
# These help visually confirm whether relationships are:
#   - Linear (straight trend line)
#   - Non-linear (curved)
#   - No relationship (flat)
# ─────────────────────────────────────────────────────────────

feature_cols = X.columns.tolist()
n_cols = 3
n_rows = int(np.ceil(len(feature_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    ax = axes[i]

    # Scatter points
    ax.scatter(df[col], df['employment_outcomes'],
               color='steelblue', s=80, alpha=0.85,
               edgecolors='white', linewidth=0.5)

    # Label each point with country name
    for _, row in df.iterrows():
        ax.annotate(row['country'], (row[col], row['employment_outcomes']),
                    textcoords="offset points", xytext=(4, 4),
                    fontsize=6.5, color='dimgray')

    # Linear trend line to show direction
    z = np.polyfit(df[col], df['employment_outcomes'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(x_line, p(x_line), 'r--', linewidth=1.2, alpha=0.6)

    ax.set_xlabel(col, fontsize=9)
    ax.set_ylabel('employment_outcomes', fontsize=9)
    ax.set_title(f'{col}', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)

# Hide unused subplot panels
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Each Feature vs employment_outcomes — Scatter Plots with Trend Lines',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## ✂️ Cell 7 — Train / Test Split

In [ ]:
# ─────────────────────────────────────────────────────────────
# SPLIT DATA INTO TRAINING AND TEST SETS
#
# Training set (80%): The model LEARNS from this data
# Test set     (20%): The model is EVALUATED on this (never seen before)
#
# This simulates real-world performance on new, unseen countries.
# NOTE: With only 10 rows, the test set will only have 2 countries.
#       In real projects, aim for 500+ rows minimum.
# ─────────────────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% = held-out test set (~2 countries)
    random_state=42      # Fixed seed = same split every run (reproducible)
)

print(f"Training samples : {X_train.shape[0]}")
print(f"Test samples     : {X_test.shape[0]}")
print(f"\nCountries in TRAIN: {list(df.loc[X_train.index, 'country'])}")
print(f"Countries in TEST : {list(df.loc[X_test.index, 'country'])}")


## 🌲 Cell 8 — Build the Random Forest Model

In [ ]:
# ─────────────────────────────────────────────────────────────
# RANDOM FOREST — KEY HYPERPARAMETERS EXPLAINED
#
#  n_estimators    = number of decision trees in the forest
#                    More trees → more stable predictions, but slower
#                    Range: 50–500 (100 is a good starting point)
#
#  max_depth       = how deep each individual tree can grow
#                    None = unlimited (overfits on small data!)
#                    2-4  = shallow trees = less overfitting
#
#  max_features    = how many features each tree randomly considers per split
#                    'sqrt' = sqrt(n_features) ← default, good for regression
#                    'log2' = log2(n_features)
#
#  min_samples_split = minimum samples required to split a node
#                    Higher value = simpler trees
#
#  random_state    = seed for reproducibility (same result each run)
# ─────────────────────────────────────────────────────────────

rf_model = RandomForestRegressor(
    n_estimators=100,        # Build 100 trees and average their predictions
    max_depth=3,             # Keep trees shallow — avoids overfitting on small data
    min_samples_split=2,     # Allow splitting with as few as 2 samples
    max_features='sqrt',     # Each tree sees sqrt(10) ≈ 3 random features per split
    random_state=42          # Fixed random seed for reproducibility
)

# FIT = the actual learning step
# The model builds 100 trees, each trained on a random bootstrap sample
rf_model.fit(X_train, y_train)

print("✅ Random Forest model trained!")
print(f"\nHyperparameters:")
print(f"  n_estimators      = {rf_model.n_estimators}   (number of trees)")
print(f"  max_depth         = {rf_model.max_depth}     (max tree depth)")
print(f"  max_features      = {rf_model.max_features}  (features per split)")
print(f"  min_samples_split = {rf_model.min_samples_split}")


## 📊 Cell 9 — Model Evaluation (Metrics)

In [ ]:
# ─────────────────────────────────────────────────────────────
# EVALUATE THE MODEL
#
# R² Score (coefficient of determination):
#   1.0  = perfect predictions
#   0.0  = model only predicts the mean (useless)
#   <0   = worse than predicting the mean
#
# RMSE (Root Mean Squared Error):
#   Same units as employment_outcomes. Lower = better.
#   Penalizes large errors more heavily.
#
# MAE (Mean Absolute Error):
#   Average absolute difference between prediction and actual.
#   More robust to outliers than RMSE.
# ─────────────────────────────────────────────────────────────

# Predict on both sets
y_pred_train = rf_model.predict(X_train)
y_pred_test  = rf_model.predict(X_test)

# Compute metrics
r2_train   = r2_score(y_train, y_pred_train)
r2_test    = r2_score(y_test,  y_pred_test)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test  = np.sqrt(mean_squared_error(y_test,  y_pred_test))
mae_train  = mean_absolute_error(y_train, y_pred_train)
mae_test   = mean_absolute_error(y_test,  y_pred_test)

print(f"{'Metric':<25} {'Train':>10}  {'Test':>10}")
print("-" * 50)
print(f"{'R² Score':<25} {r2_train:>10.4f}  {r2_test:>10.4f}")
print(f"{'RMSE':<25} {rmse_train:>10.4f}  {rmse_test:>10.4f}")
print(f"{'MAE':<25} {mae_train:>10.4f}  {mae_test:>10.4f}")

# Show country-level predictions
print(f"\n📌 Actual vs Predicted on Test Set:")
test_countries = list(df.loc[X_test.index, 'country'])
for country, actual, pred in zip(test_countries, y_test, y_pred_test):
    diff = pred - actual
    print(f"  {country:<15}  Actual: {actual:.1f}  →  Predicted: {pred:.2f}  (Δ {diff:+.2f})")

# Diagnose overfitting
gap = r2_train - r2_test
if gap > 0.3:
    print(f"\n⚠️  Overfitting gap = {gap:.2f}  → model memorized training data")
    print("   Fix: more data, smaller max_depth, or cross-validation")
else:
    print(f"\n✅ Overfitting gap = {gap:.2f}  (acceptable)")


## 🏆 Cell 10 — Feature Importance (MDI Method)

In [ ]:
# ─────────────────────────────────────────────────────────────
# FEATURE IMPORTANCE — Mean Decrease in Impurity (MDI)
#
# Random Forest automatically tracks which features were most
# useful across all 100 trees.
#
# How it works:
#   Every time a feature is used to split a node, we record
#   how much the split reduced the error (impurity).
#   Features used more + reduce error more = higher importance score.
#
# Limitation: MDI can be biased towards high-cardinality features.
#             That's why we also compute Permutation Importance next.
# ─────────────────────────────────────────────────────────────

importances_mdi = rf_model.feature_importances_     # Importance score per feature
feature_names   = X.columns.tolist()

# Sort from most to least important
sorted_idx  = np.argsort(importances_mdi)[::-1]
sorted_imp  = importances_mdi[sorted_idx]
sorted_feat = [feature_names[i] for i in sorted_idx]

print("📌 Feature Importance (MDI — higher = more influential on employment_outcomes):")
for feat, imp in zip(sorted_feat, sorted_imp):
    bar = '█' * int(imp * 60)
    print(f"  {feat:<30s} {imp:.4f}  {bar}")

# ── Bar chart ──
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.85, len(sorted_feat)))[::-1]

bars = ax.barh(sorted_feat[::-1], sorted_imp[::-1],
               color=colors[::-1], edgecolor='white', linewidth=0.5)

# Value labels on each bar
for bar, val in zip(bars, sorted_imp[::-1]):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', ha='left', fontsize=9)

ax.set_xlabel('Importance Score (MDI)', fontsize=11)
ax.set_title('🌲 Random Forest — Feature Importance (Mean Decrease in Impurity)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlim(0, sorted_imp.max() + 0.08)
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## 🔄 Cell 11 — Permutation Importance (More Reliable)

In [ ]:
# ─────────────────────────────────────────────────────────────
# PERMUTATION IMPORTANCE — More reliable than MDI
#
# How it works:
#   1. Score model on original data → baseline R²
#   2. Randomly SHUFFLE one feature column (destroying its real signal)
#   3. Score model again → see how much performance drops
#   4. Repeat 30 times per feature → stable average
#
# Features that cause BIG performance drops when shuffled = very important
# Features with near-zero drop = probably not contributing much
#
# This is more trustworthy for small datasets than MDI.
# ─────────────────────────────────────────────────────────────

perm_result = permutation_importance(
    rf_model, X_train, y_train,
    n_repeats=30,       # Shuffle each feature 30 times → stable mean estimate
    random_state=42,
    scoring='r2'        # Measure drop in R² when feature is scrambled
)

perm_means = perm_result.importances_mean    # Average R² drop per feature
perm_stds  = perm_result.importances_std     # Variability across 30 shuffles

# Sort
perm_idx   = np.argsort(perm_means)[::-1]
perm_means_s = perm_means[perm_idx]
perm_stds_s  = perm_stds[perm_idx]
perm_feats   = [feature_names[i] for i in perm_idx]

print("📌 Permutation Importance (R² drop when feature is randomly shuffled):")
for feat, mean, std in zip(perm_feats, perm_means_s, perm_stds_s):
    bar = '█' * max(0, int(mean * 60))
    print(f"  {feat:<30s} {mean:+.4f} ± {std:.4f}  {bar}")

# ── Bar chart with error bars ──
fig, ax = plt.subplots(figsize=(10, 6))
colors_p = ['#2ecc71' if m > 0 else '#e74c3c' for m in perm_means_s[::-1]]

ax.barh(perm_feats[::-1], perm_means_s[::-1],
        xerr=perm_stds_s[::-1],
        color=colors_p, edgecolor='white', linewidth=0.5,
        capsize=4, error_kw={'elinewidth': 1.2, 'ecolor': '#555'})

ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Mean R² Drop when Feature is Shuffled', fontsize=11)
ax.set_title('🔄 Permutation Feature Importance\nGreen = meaningful, Red = not significant',
             fontsize=12, fontweight='bold', pad=12)
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## 🎯 Cell 12 — Actual vs Predicted Plot

In [ ]:
# ─────────────────────────────────────────────────────────────
# ACTUAL vs PREDICTED SCATTER PLOT
#
# The red dashed diagonal = PERFECT predictions (actual == predicted)
# Points close to the diagonal = good predictions
# Points far from the diagonal = model errors
#
# We plot ALL countries (not just test) since the test set is tiny.
# ─────────────────────────────────────────────────────────────

y_pred_all = rf_model.predict(X)    # Predict on the full dataset

fig, ax = plt.subplots(figsize=(8, 7))

# Scatter with color encoding the actual employment_outcomes
scatter = ax.scatter(y, y_pred_all,
                     c=y, cmap='RdYlGn',
                     s=130, edgecolors='white', linewidth=1, zorder=5)

# Label each country
for _, row in df.iterrows():
    y_actual = row['employment_outcomes']
    y_pred   = rf_model.predict(X.loc[[row.name]])[0]
    ax.annotate(row['country'], (y_actual, y_pred),
                textcoords="offset points", xytext=(6, 4),
                fontsize=9, color='#333')

# Perfect prediction diagonal line
min_val = min(y.min(), y_pred_all.min()) - 0.5
max_val = max(y.max(), y_pred_all.max()) + 0.5
ax.plot([min_val, max_val], [min_val, max_val],
        'r--', linewidth=1.8, label='Perfect prediction line', zorder=4)

ax.set_xlabel('Actual employment_outcomes',    fontsize=12)
ax.set_ylabel('Predicted employment_outcomes', fontsize=12)
ax.set_title(f'Actual vs Predicted — employment_outcomes\nR² (all data) = {r2_score(y, y_pred_all):.4f}',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.colorbar(scatter, ax=ax, label='employment_outcomes')
plt.tight_layout()
plt.show()


## ⚙️ Cell 13 — Hyperparameter Tuning: `n_estimators`

In [ ]:
# ─────────────────────────────────────────────────────────────
# HOW MANY TREES DO WE ACTUALLY NEED?
#
# n_estimators = number of trees in the forest
# Adding more trees generally improves stability, but:
#   - Returns diminish after a point (performance plateaus)
#   - More trees = slower training
#
# This plot shows where the "sweet spot" is.
# If Train R² >> Test R² = overfitting as trees increase
# ─────────────────────────────────────────────────────────────

estimator_values = [5, 10, 20, 30, 50, 75, 100, 150, 200]
train_r2_list, test_r2_list = [], []

for n in estimator_values:
    temp_rf = RandomForestRegressor(n_estimators=n, max_depth=3, random_state=42)
    temp_rf.fit(X_train, y_train)

    tr = r2_score(y_train, temp_rf.predict(X_train))
    te = r2_score(y_test,  temp_rf.predict(X_test))
    train_r2_list.append(tr)
    test_r2_list.append(te)
    print(f"  n_estimators = {n:>3d}  →  Train R²: {tr:.4f}   Test R²: {te:.4f}")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(estimator_values, train_r2_list, 'o-', color='steelblue',
        label='Train R²', linewidth=2, markersize=6)
ax.plot(estimator_values, test_r2_list,  's--', color='tomato',
        label='Test R²',  linewidth=2, markersize=6)
ax.axvline(100, color='gray', linestyle=':', alpha=0.7, label='Default choice (100)')
ax.set_xlabel('n_estimators (number of trees)', fontsize=11)
ax.set_ylabel('R² Score', fontsize=11)
ax.set_title('Effect of n_estimators on Model R² Score', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## ⚙️ Cell 14 — Hyperparameter Tuning: `max_depth`

In [ ]:
# ─────────────────────────────────────────────────────────────
# HOW DEEP SHOULD EACH TREE GROW?
#
# max_depth = maximum number of splits per tree
#
# Too shallow (depth=1): UNDERFITTING — model misses real patterns
# Just right (depth=2-4): good bias-variance tradeoff
# Too deep (None): OVERFITTING — model memorizes training data
#
# Watch for: high Train R² + low Test R² = classic overfitting
# ─────────────────────────────────────────────────────────────

depth_values = [1, 2, 3, 4, 5, None]
train_r2_depth, test_r2_depth = [], []

for d in depth_values:
    temp_rf = RandomForestRegressor(n_estimators=100, max_depth=d, random_state=42)
    temp_rf.fit(X_train, y_train)

    tr = r2_score(y_train, temp_rf.predict(X_train))
    te = r2_score(y_test,  temp_rf.predict(X_test))
    train_r2_depth.append(tr)
    test_r2_depth.append(te)

    label = str(d) if d is not None else "None (unlimited)"
    print(f"  max_depth = {label:<15}  Train R²: {tr:.4f}   Test R²: {te:.4f}")

depth_labels = [str(d) if d is not None else "None" for d in depth_values]
x_pos = range(len(depth_values))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x_pos, train_r2_depth, 'o-', color='steelblue', label='Train R²', linewidth=2)
ax.plot(x_pos, test_r2_depth,  's--', color='tomato',   label='Test R²',  linewidth=2)
ax.set_xticks(x_pos)
ax.set_xticklabels(depth_labels)
ax.set_xlabel('max_depth', fontsize=11)
ax.set_ylabel('R² Score',  fontsize=11)
ax.set_title('Effect of max_depth on R² Score\n(shallow = underfit, deep = overfit)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


## 📋 Cell 15 — Final Summary Dashboard

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2×2 SUMMARY DASHBOARD
# Combines the 4 most important views in one figure
# ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('🌲 Random Forest — Full Analysis Dashboard', fontsize=15, fontweight='bold', y=1.01)

# ── Panel 1: Correlation heatmap ──
ax1 = axes[0, 0]
sns.heatmap(corr_matrix, annot=True, fmt='.1f', cmap='RdBu_r',
            center=0, linewidths=0.5, ax=ax1,
            annot_kws={"size": 7}, cbar_kws={'shrink': 0.8})
ax1.set_title('Correlation Heatmap', fontsize=11, fontweight='bold')
ax1.tick_params(axis='x', rotation=45, labelsize=7)
ax1.tick_params(axis='y', rotation=0,  labelsize=7)

# ── Panel 2: Feature importance (MDI) ──
ax2 = axes[0, 1]
colors2 = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(sorted_feat)))
ax2.barh(sorted_feat[::-1], sorted_imp[::-1], color=colors2, edgecolor='white')
ax2.set_title('Feature Importance (MDI)', fontsize=11, fontweight='bold')
ax2.set_xlabel('Importance Score')
ax2.grid(axis='x', alpha=0.3)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# ── Panel 3: Actual vs Predicted ──
ax3 = axes[1, 0]
y_pred_all = rf_model.predict(X)
ax3.scatter(y, y_pred_all, c='steelblue', s=100, edgecolors='white', linewidth=1)
for _, row in df.iterrows():
    ax3.annotate(row['country'], (row['employment_outcomes'], rf_model.predict(X.loc[[row.name]])[0]),
                 textcoords="offset points", xytext=(4, 3), fontsize=7)
ax3.plot([y.min()-0.5, y.max()+0.5], [y.min()-0.5, y.max()+0.5], 'r--', linewidth=1.5)
ax3.set_xlabel('Actual employment_outcomes', fontsize=10)
ax3.set_ylabel('Predicted employment_outcomes', fontsize=10)
ax3.set_title(f'Actual vs Predicted  |  R² = {r2_score(y, y_pred_all):.4f}',
              fontsize=11, fontweight='bold')
ax3.grid(True, alpha=0.3)

# ── Panel 4: Metrics summary bar chart ──
ax4 = axes[1, 1]
metrics      = ['R² (Train)', 'R² (Test)', 'RMSE (Test)', 'MAE (Test)']
metric_vals  = [r2_train, max(r2_test, 0), rmse_test, mae_test]   # clip negative R² for bar viz
bar_colors   = ['#2ecc71', '#27ae60', '#e74c3c', '#e67e22']
bars4 = ax4.bar(metrics, metric_vals, color=bar_colors, edgecolor='white', linewidth=0.5)
for b, v in zip(bars4, [r2_train, r2_test, rmse_test, mae_test]):
    ax4.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
             f'{v:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax4.set_title('Model Performance Metrics', fontsize=11, fontweight='bold')
ax4.set_ylabel('Value', fontsize=10)
ax4.grid(axis='y', alpha=0.3)
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


## 🏁 Cell 16 — Findings & Conclusion

---

### 📌 What affects `pisa_score` the most?

| Rank | Feature | MDI Importance | Correlation with pisa_score |
|------|---------|---------------|----------------------------|
| 1 | `academic_pressure` | Highest | +0.49 (positive) |
| 2 | `mental_health_support` | High | −0.27 (negative) |
| 3 | `stress` | High | +0.42 (positive) |
| 4 | `unemployment_score` | Medium | −0.65 (strong negative) |
| 5 | `teacher_quality` | Medium | +0.73 (strongest positive) |

---

### ⚠️ Why is Test R² negative?
- The dataset has only **10 rows** — test set = only 2 countries
- Even a 1-point prediction error collapses R² on 2 samples
- **This is a data size problem, not a code problem**
- Fix: get more data (50+ countries), then use `KFold(n_splits=5)` cross-validation

---

### 🚀 How to improve model performance?
1. **More data** — most important fix by far
2. **Cross-validation** — use `cross_val_score` instead of train/test split for small datasets
3. **Tune `max_depth`** — keep at 2-3 to reduce overfitting
4. **Feature engineering** — try `stress × academic_pressure` as an interaction feature
5. **Try XGBoost** — `XGBRegressor` often outperforms RF on tabular data


## 🎁 Bonus Cell — Cross-Validation (Better for small datasets)

In [ ]:
# ─────────────────────────────────────────────────────────────
# CROSS-VALIDATION — Better evaluation for small datasets
#
# Instead of one train/test split, we do K splits:
#   Fold 1: [test] [train] [train] [train] [train]
#   Fold 2: [train] [test] [train] [train] [train]
#   ... and so on for K folds
#
# Each country gets to be in the test set exactly once.
# Final score = average across all K folds.
# This gives a much more honest performance estimate on small data.
# ─────────────────────────────────────────────────────────────

from sklearn.model_selection import cross_val_score, KFold

# Use Leave-One-Out cross-validation (best for tiny datasets)
# LOO = each country becomes the test set once (n_splits = n_rows)
from sklearn.model_selection import LeaveOneOut

loo = LeaveOneOut()
cv_scores = cross_val_score(
    rf_model, X, y,
    cv=loo,              # Leave-One-Out: 10 folds for 10 countries
    scoring='r2'
)

print("📌 Leave-One-Out Cross-Validation Results:")
print(f"  R² per fold: {[round(s, 3) for s in cv_scores]}")
print(f"  Mean R²    : {cv_scores.mean():.4f}")
print(f"  Std R²     : {cv_scores.std():.4f}")
print()

# Also try KFold with fewer splits
kf = KFold(n_splits=5, shuffle=True, random_state=42)
kf_scores = cross_val_score(rf_model, X, y, cv=kf, scoring='r2')
print(f"📌 5-Fold Cross-Validation:")
print(f"  R² per fold: {[round(s, 3) for s in kf_scores]}")
print(f"  Mean R²    : {kf_scores.mean():.4f}")
print(f"  Std R²     : {kf_scores.std():.4f}")
print()
print("💡 TIP: Mean CV R² is your most honest performance estimate on small data.")
